In [1]:
# ══════════════════════════════════════════════════════════════════════════════
#  CELL 1 — MODEL DEFINITIONS
# ══════════════════════════════════════════════════════════════════════════════
 
import os, gc, math, random, warnings
warnings.filterwarnings("ignore")
 
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models
 
SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)
random.seed(SEED)
 
PATCH_SIZE = 64
STEP       = 32
N_RADAR    = 3
N_OPTICAL  = 7        # ← PATH A: was 4, now 7 (adds NDVI, NDWI, BSI)
AUTO       = tf.data.AUTOTUNE
 
# ── Architecture (identical to v6) ────────────────────────────────────────────
 
def cbam_attention(inputs, reduction_ratio=8):
    filters  = int(inputs.shape[-1])
    avg_pool = layers.GlobalAveragePooling2D(keepdims=True)(inputs)
    max_pool = layers.GlobalMaxPooling2D(keepdims=True)(inputs)
    shared_1 = layers.Dense(max(filters // reduction_ratio, 1),
                             activation="relu", use_bias=False)
    shared_2 = layers.Dense(filters, use_bias=False)
    ch_att   = layers.Activation("sigmoid")(
        layers.Add()([shared_2(shared_1(avg_pool)),
                      shared_2(shared_1(max_pool))])
    )
    x       = layers.Multiply()([inputs, ch_att])
    mean_sp = layers.Lambda(lambda z: tf.reduce_mean(z, axis=-1, keepdims=True))(x)
    max_sp  = layers.Lambda(lambda z: tf.reduce_max(z, axis=-1, keepdims=True))(x)
    sp      = layers.Conv2D(1, 7, padding="same", activation="sigmoid")(
        layers.Concatenate(axis=-1)([mean_sp, max_sp])
    )
    return layers.Multiply()([x, sp])
 
 
def residual_block(x, filters, kernel_size=3):
    sc = x
    x  = layers.Conv2D(filters, kernel_size, padding="same")(x)
    x  = layers.BatchNormalization(momentum=0.95)(x)
    x  = layers.Activation("relu")(x)
    x  = layers.Conv2D(filters, kernel_size, padding="same")(x)
    x  = layers.BatchNormalization(momentum=0.95)(x)
    if int(sc.shape[-1]) != filters:
        sc = layers.Conv2D(filters, 1, padding="same")(sc)
    x  = layers.Add()([x, sc])
    return layers.Activation("relu")(x)
 
 
def build_super_model(filters=48):
    inp_r = layers.Input(shape=(PATCH_SIZE, PATCH_SIZE, N_RADAR),   name="radar")
    inp_o = layers.Input(shape=(PATCH_SIZE, PATCH_SIZE, N_OPTICAL), name="optical")
    # ↑ N_OPTICAL=7 now — model graph automatically wider here
 
    r = layers.Conv2D(filters, 3, padding="same", activation="relu")(inp_r)
    r = residual_block(r, filters)
    r = cbam_attention(r)
 
    o = layers.Conv2D(filters, 3, padding="same", activation="relu")(inp_o)
    o = residual_block(o, filters)
    o = cbam_attention(o)
 
    x = layers.Concatenate()([r, o])
    x = residual_block(x, filters * 2)
    x = layers.SpatialDropout2D(0.15)(x)
    x = layers.Conv2D(filters * 2, 3, padding="same", activation="relu")(x)
    x = layers.SpatialDropout2D(0.10)(x)
    x = layers.Conv2D(filters, 3, padding="same", activation="relu")(x)
    out = layers.Conv2D(1, 1, padding="same", activation="linear",
                        name="smap_value")(x)
 
    return models.Model(inputs={"radar": inp_r, "optical": inp_o}, outputs=out)
 
 
# ── Metrics (identical to v6) ─────────────────────────────────────────────────
 
class MaskedRMSE(tf.keras.metrics.Metric):
    def __init__(self, name="rmse", **kwargs):
        super().__init__(name=name, **kwargs)
        self.ss = self.add_weight(name="ss", initializer="zeros")
        self.ct = self.add_weight(name="ct", initializer="zeros")
 
    def update_state(self, yt, yp, sample_weight=None):
        yv = yt[:, :, :, 0:1]
        ym = yt[:, :, :, 1:2]
        self.ss.assign_add(tf.reduce_sum(tf.square(yv - yp) * ym))
        self.ct.assign_add(tf.reduce_sum(ym))
 
    def result(self):
        return tf.sqrt(self.ss / (self.ct + 1e-8))
 
    def reset_states(self):
        self.ss.assign(0.0)
        self.ct.assign(0.0)
 
 
class MaskedPearson(tf.keras.metrics.Metric):
    def __init__(self, name="pearson", **kwargs):
        super().__init__(name=name, **kwargs)
        self.sum_t  = self.add_weight(name="sum_t",  initializer="zeros")
        self.sum_p  = self.add_weight(name="sum_p",  initializer="zeros")
        self.sum_tt = self.add_weight(name="sum_tt", initializer="zeros")
        self.sum_pp = self.add_weight(name="sum_pp", initializer="zeros")
        self.sum_tp = self.add_weight(name="sum_tp", initializer="zeros")
        self.ct     = self.add_weight(name="ct",     initializer="zeros")
 
    def update_state(self, yt, yp, sample_weight=None):
        yv   = yt[:, :, :, 0:1]
        ym   = yt[:, :, :, 1:2]
        mask = tf.cast(ym > 0.5, tf.bool)
        t    = tf.boolean_mask(yv, mask)
        p    = tf.boolean_mask(yp, mask)
        self.sum_t.assign_add(tf.reduce_sum(t))
        self.sum_p.assign_add(tf.reduce_sum(p))
        self.sum_tt.assign_add(tf.reduce_sum(t * t))
        self.sum_pp.assign_add(tf.reduce_sum(p * p))
        self.sum_tp.assign_add(tf.reduce_sum(t * p))
        self.ct.assign_add(tf.cast(tf.size(t), tf.float32))
 
    def result(self):
        n   = self.ct + 1e-8
        mt  = self.sum_t / n
        mp  = self.sum_p / n
        cov = self.sum_tp / n - mt * mp
        vt  = self.sum_tt / n - mt * mt
        vp  = self.sum_pp / n - mp * mp
        return cov / (tf.sqrt(vt * vp) + 1e-8)
 
    def reset_states(self):
        for v in self.variables:
            v.assign(0.0)
 
 
# ── LR Schedule (identical to v6) ─────────────────────────────────────────────
 
class ConstrainedCyclicDecay(tf.keras.optimizers.schedules.LearningRateSchedule):
    def __init__(self, peak_lr=1e-5, min_lr=1e-7, cycle_steps=1000):
        super().__init__()
        self.peak_lr    = float(peak_lr)
        self.min_lr     = float(min_lr)
        self.cycle_steps = float(cycle_steps)
 
    def __call__(self, step):
        step   = tf.cast(step, tf.float32)
        period = self.cycle_steps * 2.0
        x      = tf.math.mod(step, period)
        frac   = tf.where(
            x <= self.cycle_steps,
            x / tf.maximum(self.cycle_steps, 1.0),
            (period - x) / tf.maximum(self.cycle_steps, 1.0),
        )
        frac = tf.clip_by_value(frac, 0.0, 1.0)
        return self.min_lr + (self.peak_lr - self.min_lr) * frac
 
    def get_config(self):
        return {"peak_lr": self.peak_lr,
                "min_lr":  self.min_lr,
                "cycle_steps": self.cycle_steps}
 
 
# ── Loss Model — v7 adds physics penalty ──────────────────────────────────────
 
class VarianceAwareMaskedModel(tf.keras.Model):
    """
    Identical to v6 except _physics_penalty() is new (Path D).
 
    Physics penalty:
        Penalises predictions outside physical bounds [PHYS_LO, PHYS_HI] m³/m³.
        Applied in Z-score space: bounds are converted with SMAP_MEAN/SMAP_STD.
        Weight lambda_phys=0.01 — small enough not to overpower MSE.
 
    All other loss components (MSE, std penalty, gradient penalty) unchanged.
    """
 
    # Physical moisture bounds in m³/m³
    PHYS_LO = 0.0
    PHYS_HI = 0.45
 
    def __init__(self, base_model,
                 lambda_std=0.01,
                 lambda_grad=0.01,
                 lambda_phys=0.01,   # ← PATH D: new parameter
                 smap_mean=0.0,      # set after pipeline builds stats
                 smap_std=1.0,
                 **kwargs):
        super().__init__(**kwargs)
        self.base_model  = base_model
        self.lambda_std  = float(lambda_std)
        self.lambda_grad = float(lambda_grad)
        self.lambda_phys = float(lambda_phys)
        self.smap_mean   = float(smap_mean)
        self.smap_std    = float(smap_std)
 
        self.loss_m = tf.keras.metrics.Mean(name="loss")
        self.rmse_m = MaskedRMSE(name="rmse")
        self.prsn_m = MaskedPearson(name="pearson")
 
    def call(self, inputs, training=False):
        return self.base_model(inputs, training=training)
 
    # ── individual loss components ─────────────────────────────────────────────
 
    def _masked_mse(self, y_val, y_pred, mask):
        return (tf.reduce_sum(tf.square(y_val - y_pred) * mask)
                / (tf.reduce_sum(mask) + 1e-8))
 
    def _std_penalty(self, y_val, y_pred, mask):
        if self.lambda_std == 0:
            return 0.0
        count      = tf.reduce_sum(mask, axis=[1, 2, 3], keepdims=True)
        mean_p     = (tf.reduce_sum(y_pred * mask, axis=[1, 2, 3], keepdims=True)
                      / (count + 1e-8))
        var_p      = (tf.reduce_sum(tf.square(y_pred - mean_p) * mask,
                                    axis=[1, 2, 3], keepdims=True)
                      / (count + 1e-8))
        std_p      = tf.sqrt(var_p + 1e-8)
        mean_l     = (tf.reduce_sum(y_val * mask, axis=[1, 2, 3], keepdims=True)
                      / (count + 1e-8))
        var_l      = (tf.reduce_sum(tf.square(y_val - mean_l) * mask,
                                    axis=[1, 2, 3], keepdims=True)
                      / (count + 1e-8))
        std_l      = tf.stop_gradient(tf.sqrt(var_l + 1e-8))
        valid_patch = tf.cast(count > 16.0, tf.float32)
        penalty    = tf.square(std_p - std_l) * valid_patch
        return (tf.reduce_sum(penalty)
                / (tf.reduce_sum(valid_patch) + 1e-8))
 
    def _gradient_penalty(self, y_val, y_pred, mask):
        if self.lambda_grad == 0:
            return 0.0
        gx_p  = y_pred[:, :, 1:, :] - y_pred[:, :, :-1, :]
        gx_y  = y_val[:,  :, 1:, :] - y_val[:,  :, :-1, :]
        gx_m  = mask[:,   :, 1:, :] * mask[:,    :, :-1, :]
        gy_p  = y_pred[:, 1:, :, :] - y_pred[:, :-1, :, :]
        gy_y  = y_val[:,  1:, :, :] - y_val[:,  :-1, :, :]
        gy_m  = mask[:,   1:, :, :] * mask[:,    :-1, :, :]
        gx_loss = (tf.reduce_sum(tf.square(gx_p - gx_y) * gx_m)
                   / (tf.reduce_sum(gx_m) + 1e-8))
        gy_loss = (tf.reduce_sum(tf.square(gy_p - gy_y) * gy_m)
                   / (tf.reduce_sum(gy_m) + 1e-8))
        return 0.5 * (gx_loss + gy_loss)
 
    def _physics_penalty(self, y_pred):
        """
        PATH D: Penalise predictions outside physical moisture bounds.
 
        y_pred is in Z-score space.  Convert bounds to Z-score space once.
        Penalty = mean squared violation outside [lo_z, hi_z].
 
        lo_z = (0.000 - SMAP_MEAN) / SMAP_STD
        hi_z = (0.450 - SMAP_MEAN) / SMAP_STD
 
        With SMAP_MEAN≈0.066, SMAP_STD≈0.028:
            lo_z ≈ -2.36   (0.0 m³/m³)
            hi_z ≈ +13.71  (0.45 m³/m³ — very permissive upper bound)
 
        In practice only the lower bound fires (negative predictions).
        The upper bound protects against extreme outliers.
        """
        if self.lambda_phys == 0:
            return 0.0
 
        lo_z = (self.PHYS_LO - self.smap_mean) / (self.smap_std + 1e-8)
        hi_z = (self.PHYS_HI - self.smap_mean) / (self.smap_std + 1e-8)
 
        # Violation below lower bound (negative moisture)
        below = tf.square(tf.minimum(y_pred - lo_z, 0.0))
        # Violation above upper bound (> 0.45 m³/m³)
        above = tf.square(tf.maximum(y_pred - hi_z, 0.0))
 
        return tf.reduce_mean(below + above)
 
    def _loss(self, y_val, y_pred, mask):
        mse    = self._masked_mse(y_val, y_pred, mask)
        std_p  = self._std_penalty(y_val, y_pred, mask)
        grad_p = self._gradient_penalty(y_val, y_pred, mask)
        phys_p = self._physics_penalty(y_pred)          # ← PATH D
        return (mse
                + self.lambda_std  * std_p
                + self.lambda_grad * grad_p
                + self.lambda_phys * phys_p)
 
    # ── train / test steps ────────────────────────────────────────────────────
 
    def train_step(self, data):
        bx, by = data
        yv, ym = by["smap_value"], by["valid_mask"]
        with tf.GradientTape() as tape:
            yp   = self(bx, training=True)
            loss = self._loss(yv, yp, ym)
        grads, _ = tf.clip_by_global_norm(
            tape.gradient(loss, self.trainable_variables), 1.0)
        self.optimizer.apply_gradients(
            zip(grads, self.trainable_variables))
        combined = tf.concat([yv, ym], -1)
        self.loss_m.update_state(loss)
        self.rmse_m.update_state(combined, yp)
        self.prsn_m.update_state(combined, yp)
        return {m.name: m.result() for m in self.metrics}
 
    def test_step(self, data):
        bx, by = data
        yv, ym = by["smap_value"], by["valid_mask"]
        yp     = self(bx, training=False)
        loss   = self._loss(yv, yp, ym)
        combined = tf.concat([yv, ym], -1)
        self.loss_m.update_state(loss)
        self.rmse_m.update_state(combined, yp)
        self.prsn_m.update_state(combined, yp)
        return {m.name: m.result() for m in self.metrics}
 
    @property
    def metrics(self):
        return [self.loss_m, self.rmse_m, self.prsn_m]
 
 
# ── Diagnostic callback (identical to v6) ─────────────────────────────────────
 
class DiagnosticCallback(tf.keras.callbacks.Callback):
    def __init__(self, val_ds):
        super().__init__()
        self.val_ds = val_ds
 
    def on_epoch_end(self, epoch, logs=None):
        pl, tl, ml = [], [], []
        for x, y in self.val_ds:
            p = self.model(x, training=False)
            pl.append(p.numpy())
            tl.append(y["smap_value"].numpy())
            ml.append(y["valid_mask"].numpy())
        pf     = np.concatenate(pl).flatten()
        tf_arr = np.concatenate(tl).flatten()
        mf     = np.concatenate(ml).flatten()
        vp, vl = pf[mf > 0.5], tf_arr[mf > 0.5]
 
        pred_std  = float(vp.std()) if len(vp) > 0 else 0.0
        label_std = float(vl.std()) if len(vl) > 0 else 0.0
        pr        = (float(np.corrcoef(vl, vp)[0, 1])
                     if len(vp) > 10 and pred_std > 1e-6 else 0.0)
 
        # Physics violation check (in raw m³/m³ space)
        raw_preds = vp * self.model.smap_std + self.model.smap_mean
        n_below   = int((raw_preds < 0.0).sum())
        n_above   = int((raw_preds > 0.45).sum())
 
        try:
            lr_obj = self.model.optimizer.learning_rate
            lr = float(lr_obj(self.model.optimizer.iterations))
        except Exception:
            lr = 0.0
 
        print(
            f"  [EP {epoch+1}] VAL => "
            f"pred_std={pred_std:.4f} label_std={label_std:.4f} "
            f"FULL_VAL_PEARSON={pr:+.4f} lr={lr:.2e} "
            f"| physics violations: below={n_below} above={n_above}"
        )
        gc.collect()
 
 
print("Cell 1 ready — Desert SuperModel v7 definitions loaded.")
print(f"  N_RADAR   = {N_RADAR}  (VV, VH, VV/VH)")
print(f"  N_OPTICAL = {N_OPTICAL}  (B4, B8, B11, B12, NDVI, NDWI, BSI)  ← PATH A")
print(f"  Physics penalty: [0.0, 0.45] m³/m³                              ← PATH D")
 

2026-05-06 03:52:38.701714: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1778039558.890427      57 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1778039558.945156      57 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1778039559.420003      57 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1778039559.420048      57 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1778039559.420051      57 computation_placer.cc:177] computation placer alr

Cell 1 ready — Desert SuperModel v7 definitions loaded.
  N_RADAR   = 3  (VV, VH, VV/VH)
  N_OPTICAL = 7  (B4, B8, B11, B12, NDVI, NDWI, BSI)  ← PATH A
  Physics penalty: [0.0, 0.45] m³/m³                              ← PATH D


In [2]:
# CELL 2: ALL-DESERT SPATIAL 80/20 PIPELINE, STORAGE-SAFE

import os
import gc
import json
import random
from glob import glob

import numpy as np
import tensorflow as tf
import rasterio

SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)
random.seed(SEED)

PATCH_SIZE = 64
STEP = 32
N_FEATS = 10   # ← (7 → 10)
AUTO = tf.data.AUTOTUNE

GEOTIFF_DIR = "/kaggle/input/datasets/waleedeltanany/geo-tiffs/ds"

ALL_DESERT_TIFS = [
    "Kharga_Winter_SuperModel_14B.tif",
    "Dakhla_Winter_SuperModel_14B.tif",
    "Dakhla_Spring_SuperModel_14B.tif",
    "Bahariya_Winter_SuperModel_14B.tif",
    "Bahariya_Spring_SuperModel_14B.tif",
    "Farafra_Spring_SuperModel_14B.tif",
    "Siwa_Winter_SuperModel_14B.tif",
    "Siwa_Spring_SuperModel_14B.tif",
]

# ───────────────────────────────
# ONLY CHANGES (v7)
# ───────────────────────────────
OPTICAL_IDX     = [3, 4, 5, 6, 7, 8, 9]   # ← updated
RAW_PHYSICS_IDX = [0, 1, 2] + OPTICAL_IDX # ← updated
# ───────────────────────────────

RAW_PHYSICS_BANDS = [i + 1 for i in RAW_PHYSICS_IDX]

BASE_DIR = "/kaggle/working"
RUN_NAME = "all_desert_spatial80"
TFRECORDS_DIR = os.path.join(BASE_DIR, f"tfrecords_{RUN_NAME}")
CHECKPOINTS_DIR = os.path.join(BASE_DIR, f"checkpoints_{RUN_NAME}")
LOGS_DIR = os.path.join(BASE_DIR, f"logs_{RUN_NAME}")
META_DIR = os.path.join(BASE_DIR, f"meta_{RUN_NAME}")

for d in [TFRECORDS_DIR, CHECKPOINTS_DIR, LOGS_DIR, META_DIR]:
    os.makedirs(d, exist_ok=True)

NORM_STATS_PATH = os.path.join(META_DIR, "feature_stats_rawphysics7.npz")
SMAP_STATS_PATH = os.path.join(META_DIR, "smap_stats_trainval_spatial80.npz")

def region_name(tif_name):
    return tif_name.replace("_SuperModel_14B.tif", "")

def patch_positions(length, patch_size=64, step=32):
    pos = list(range(0, max(length - patch_size + 1, 1), step))
    last = length - patch_size
    if last >= 0 and (len(pos) == 0 or pos[-1] != last):
        pos.append(last)
    return pos

def is_train_patch(x, width, patch_size=64, frac=0.80):
    center_x = x + patch_size / 2.0
    return center_x < (width * frac)

def _bytes_feature(value):
    return tf.train.Feature(bytes_list=tf.train.BytesList(value=[value]))

def serialize_example(feat, label, mask):
    feat = feat.astype(np.float16)
    label = label.astype(np.float16)
    mask = mask.astype(np.float16)

    feature = {
        "feat": _bytes_feature(tf.io.serialize_tensor(feat).numpy()),
        "label": _bytes_feature(tf.io.serialize_tensor(label).numpy()),
        "mask": _bytes_feature(tf.io.serialize_tensor(mask).numpy()),
    }
    return tf.train.Example(features=tf.train.Features(feature=feature)).SerializeToString()

def fill_nans_per_channel(feat):
    feat = feat.copy()
    for c in range(feat.shape[-1]):
        ch = feat[:, :, c]
        m = np.isnan(ch)
        if m.any():
            med = np.nanmedian(ch)
            ch[m] = med if np.isfinite(med) else 0.0
            feat[:, :, c] = ch
    return feat

def compute_feature_stats(trainval_tifs, max_samples_per_region=250000):
    if os.path.exists(NORM_STATS_PATH):
        s = np.load(NORM_STATS_PATH)
        mean = s["mean"].astype(np.float32)
        std = s["std"].astype(np.float32)
        print(f"Loaded feature stats from {NORM_STATS_PATH}")
        return mean, std

    print("Computing raw feature statistics...")
    vals = [[] for _ in range(len(RAW_PHYSICS_IDX))]

    for tif in trainval_tifs:
        path = os.path.join(GEOTIFF_DIR, tif)
        print("  ->", tif)
        with rasterio.open(path) as src:
            arr = src.read(RAW_PHYSICS_BANDS).astype(np.float32)
        arr = np.transpose(arr, (1, 2, 0))
        H, W, C = arr.shape

        x_cut = int(W * 0.80)
        train_part = arr[:, :x_cut, :].reshape(-1, C)

        if train_part.shape[0] > max_samples_per_region:
            idx = np.random.choice(train_part.shape[0], max_samples_per_region, replace=False)
            train_part = train_part[idx]

        for c in range(C):
            ch = train_part[:, c]
            ch = ch[np.isfinite(ch)]
            if ch.size:
                vals[c].append(ch)

        del arr, train_part
        gc.collect()

    mean = []
    std = []
    for c in range(len(RAW_PHYSICS_IDX)):
        ch = np.concatenate(vals[c], axis=0)
        m = float(np.mean(ch))
        s = float(np.std(ch))
        if not np.isfinite(s) or s < 1e-6:
            s = 1.0
        mean.append(m)
        std.append(s)

    mean = np.asarray(mean, dtype=np.float32)
    std = np.asarray(std, dtype=np.float32)
    np.savez(NORM_STATS_PATH, mean=mean, std=std)
    print(f"Saved feature stats to {NORM_STATS_PATH}")
    return mean, std

def extract_patches_tfrecord(tif_filename, split_name, feature_mean, feature_std, patch_size=64, step=32, shard_size=512):
    tif_path = os.path.join(GEOTIFF_DIR, tif_filename)
    region = region_name(tif_filename)
    out_dir = os.path.join(TFRECORDS_DIR, split_name, region)
    os.makedirs(out_dir, exist_ok=True)

    meta_path = os.path.join(out_dir, "_meta.json")
    if os.path.exists(meta_path):
        meta = json.load(open(meta_path, "r"))
        print(f"Already extracted: {region} ({meta['patches']} patches, {meta['shards']} shards)")
        return out_dir, int(meta["patches"])

    with rasterio.open(tif_path) as src:
        arr = src.read().astype(np.float32)

    arr = np.transpose(arr, (1, 2, 0))
    H, W, _ = arr.shape

    ys = patch_positions(H, patch_size, step)
    xs = patch_positions(W, patch_size, step)

    examples = []
    patch_count = 0
    shard_idx = 0

    def flush_examples():
        nonlocal examples, shard_idx
        if not examples:
            return
        shard_path = os.path.join(out_dir, f"part-{shard_idx:05d}.tfrec")
        with tf.io.TFRecordWriter(
            shard_path,
            options=tf.io.TFRecordOptions(compression_type="GZIP")
        ) as w:
            for ex in examples:
                w.write(ex)
        examples = []
        shard_idx += 1

    for y in ys:
        for x in xs:
            train_flag = is_train_patch(x, W, patch_size, 0.80)
            if split_name == "train" and not train_flag:
                continue
            if split_name == "val" and train_flag:
                continue

            patch = arr[y:y+patch_size, x:x+patch_size, :]

            label = patch[:, :, 13].astype(np.float32)
            nan_mask = np.isnan(label)
            valid_mask = ((~nan_mask) & (label > 0.001)).astype(np.float32)

            if valid_mask.mean() < 0.10:
                continue

            label_fill = label.copy()
            label_fill[nan_mask] = 0.0
            label_fill[valid_mask < 0.5] = 0.0

            feat = patch[:, :, RAW_PHYSICS_IDX].astype(np.float32)
            feat = fill_nans_per_channel(feat)
            feat = (feat - feature_mean) / feature_std

            feat = feat.astype(np.float32)
            label_fill = label_fill[:, :, None].astype(np.float32)
            valid_mask = valid_mask[:, :, None].astype(np.float32)

            examples.append(serialize_example(feat, label_fill, valid_mask))
            patch_count += 1

            if len(examples) >= shard_size:
                flush_examples()

    flush_examples()

    meta = {
        "region": region,
        "patches": int(patch_count),
        "shards": int(shard_idx),
        "split": split_name,
    }
    with open(meta_path, "w") as f:
        json.dump(meta, f)

    return out_dir, patch_count

def parse_tfrecord_fn(example_proto):
    desc = {
        "feat": tf.io.FixedLenFeature([], tf.string),
        "label": tf.io.FixedLenFeature([], tf.string),
        "mask": tf.io.FixedLenFeature([], tf.string),
    }
    ex = tf.io.parse_single_example(example_proto, desc)

    feat = tf.io.parse_tensor(ex["feat"], out_type=tf.float16)
    label = tf.io.parse_tensor(ex["label"], out_type=tf.float16)
    mask = tf.io.parse_tensor(ex["mask"], out_type=tf.float16)

    feat = tf.cast(feat, tf.float32)
    label = tf.cast(label, tf.float32)
    mask = tf.cast(mask, tf.float32)

    feat.set_shape([PATCH_SIZE, PATCH_SIZE, N_FEATS])
    label.set_shape([PATCH_SIZE, PATCH_SIZE, 1])
    mask.set_shape([PATCH_SIZE, PATCH_SIZE, 1])

    x = {
        "radar": feat[:, :, :3],
        "optical": feat[:, :, 3:],
    }
    y = {
        "smap_value": label,
        "valid_mask": mask,
    }
    return x, y

def build_region_dataset(region_dir, training=True):
    files = sorted(glob(os.path.join(region_dir, "*.tfrec")))
    ds = tf.data.TFRecordDataset(files, compression_type="GZIP", num_parallel_reads=AUTO)
    ds = ds.map(parse_tfrecord_fn, num_parallel_calls=AUTO)
    if training:
        ds = ds.shuffle(2048, reshuffle_each_iteration=True).repeat()
    return ds

def compute_smap_stats_from_train_dirs(train_dirs, sample_cap_per_region=50000):
    if os.path.exists(SMAP_STATS_PATH):
        s = np.load(SMAP_STATS_PATH)
        print(f"Loaded SMAP stats: mean={float(s['mean']):.5f} std={float(s['std']):.5f}")
        return float(s["mean"]), float(s["std"])

    print("Computing SMAP label statistics from TFRecords...")
    all_vals = []

    for region, d in train_dirs.items():
        vals = []
        files = sorted(glob(os.path.join(d, "*.tfrec")))
        ds = tf.data.TFRecordDataset(files, compression_type="GZIP")
        ds = ds.map(parse_tfrecord_fn, num_parallel_calls=AUTO)

        taken = 0
        for _, y in ds:
            label = y["smap_value"].numpy().reshape(-1)
            mask = y["valid_mask"].numpy().reshape(-1)
            valid = label[(mask > 0.5) & (label > 0.001)]
            if valid.size:
                vals.append(valid)
                taken += valid.size
            if taken >= sample_cap_per_region:
                break

        vals = np.concatenate(vals, axis=0)[:sample_cap_per_region]
        all_vals.append(vals)
        print(f"  {region:<24}: n={len(vals):6d}  mean={vals.mean():.5f}  std={vals.std():.5f}")

    all_vals = np.concatenate(all_vals, axis=0)
    smap_mean = float(np.mean(all_vals))
    smap_std = float(np.std(all_vals))
    if not np.isfinite(smap_std) or smap_std < 1e-6:
        smap_std = 1.0

    np.savez(SMAP_STATS_PATH, mean=smap_mean, std=smap_std)
    print(f"  SMAP mean={smap_mean:.5f}  std={smap_std:.5f}")
    return smap_mean, smap_std

FEATURE_MEAN, FEATURE_STD = compute_feature_stats(ALL_DESERT_TIFS)

print("\nExtracting train/val patches...")
train_region_dirs = {}
val_region_dirs = {}
train_counts = {}
val_counts = {}

for tif in ALL_DESERT_TIFS:
    print(f"  -> {tif:<45}", end="")
    train_dir, n_train = extract_patches_tfrecord(tif, "train", FEATURE_MEAN, FEATURE_STD, PATCH_SIZE, STEP)
    val_dir, n_val = extract_patches_tfrecord(tif, "val", FEATURE_MEAN, FEATURE_STD, PATCH_SIZE, STEP)
    region = region_name(tif)
    train_region_dirs[region] = train_dir
    val_region_dirs[region] = val_dir
    train_counts[region] = n_train
    val_counts[region] = n_val
    print(f" train={n_train} val={n_val}")

SMAP_MEAN, SMAP_STD = compute_smap_stats_from_train_dirs(train_region_dirs)

def renorm_targets(x, y):
    yv = (y["smap_value"] - SMAP_MEAN) / SMAP_STD
    return x, {"smap_value": yv, "valid_mask": y["valid_mask"]}

BATCH_SIZE = 32

print("\nBuilding balanced train dataset...")
region_train_datasets = []
region_names = []

for region, d in train_region_dirs.items():
    ds = build_region_dataset(d, training=True)
    ds = ds.map(renorm_targets, num_parallel_calls=AUTO)
    region_train_datasets.append(ds)
    region_names.append(region)

train_ds = tf.data.Dataset.sample_from_datasets(
    region_train_datasets,
    weights=[1.0 / len(region_train_datasets)] * len(region_train_datasets),
    stop_on_empty_dataset=False,
)
train_ds = train_ds.batch(BATCH_SIZE, drop_remainder=True).prefetch(AUTO)

print("Building validation dataset...")
val_files = []
for region, d in val_region_dirs.items():
    files = sorted(glob(os.path.join(d, "*.tfrec")))
    val_files.extend(files)

val_ds = tf.data.TFRecordDataset(val_files, compression_type="GZIP", num_parallel_reads=AUTO)
val_ds = val_ds.map(parse_tfrecord_fn, num_parallel_calls=AUTO)
val_ds = val_ds.map(renorm_targets, num_parallel_calls=AUTO)
val_ds = val_ds.batch(BATCH_SIZE).prefetch(AUTO)

TOTAL_TRAIN_PATCHES = int(sum(train_counts.values()))
STEPS_PER_EPOCH = TOTAL_TRAIN_PATCHES // BATCH_SIZE

print("\nTrain patch counts:")
for region in region_names:
    print(f"  {region:<24}: {train_counts[region]:6d}")

print("\nVal patch counts:")
for region in region_names:
    print(f"  {region:<24}: {val_counts[region]:6d}")

print(f"\nSMAP_MEAN={SMAP_MEAN:.5f}  SMAP_STD={SMAP_STD:.5f}")
print(f"steps_per_epoch={STEPS_PER_EPOCH}")
print("Cell 2 ready.")
NORM_MEAN_1D = FEATURE_MEAN
NORM_STD_1D  = FEATURE_STD

Computing raw feature statistics...
  -> Kharga_Winter_SuperModel_14B.tif
  -> Dakhla_Winter_SuperModel_14B.tif
  -> Dakhla_Spring_SuperModel_14B.tif
  -> Bahariya_Winter_SuperModel_14B.tif
  -> Bahariya_Spring_SuperModel_14B.tif
  -> Farafra_Spring_SuperModel_14B.tif
  -> Siwa_Winter_SuperModel_14B.tif
  -> Siwa_Spring_SuperModel_14B.tif
Saved feature stats to /kaggle/working/meta_all_desert_spatial80/feature_stats_rawphysics7.npz

Extracting train/val patches...
  -> Kharga_Winter_SuperModel_14B.tif             

2026-05-06 03:57:17.439337: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


 train=22268 val=5488
  -> Dakhla_Winter_SuperModel_14B.tif              train=17954 val=4315
  -> Dakhla_Spring_SuperModel_14B.tif              train=17954 val=4315
  -> Bahariya_Winter_SuperModel_14B.tif            train=17486 val=4178
  -> Bahariya_Spring_SuperModel_14B.tif            train=17486 val=4178
  -> Farafra_Spring_SuperModel_14B.tif             train=22172 val=5291
  -> Siwa_Winter_SuperModel_14B.tif                train=13262 val=3134
  -> Siwa_Spring_SuperModel_14B.tif                train=13262 val=3134
Computing SMAP label statistics from TFRecords...
  Kharga_Winter           : n= 50000  mean=0.04816  std=0.00004
  Dakhla_Winter           : n= 50000  mean=0.04332  std=0.00015
  Dakhla_Spring           : n= 50000  mean=0.04279  std=0.00016
  Bahariya_Winter         : n= 50000  mean=0.05567  std=0.00016
  Bahariya_Spring         : n= 50000  mean=0.05052  std=0.00010
  Farafra_Spring          : n= 50000  mean=0.05037  std=0.00001
  Siwa_Winter             : n= 50000  me

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
#  CELL 3 — TRAINING
#  Replace the model instantiation block only. Everything else is identical.
# ══════════════════════════════════════════════════════════════════════════════
 

 
# ── v7 model: 7 optical channels + physics penalty ────────────────────────────
model_v7 = VarianceAwareMaskedModel(
    build_super_model(),          # N_OPTICAL=7 is read from global
    lambda_std  = 0.01,
    lambda_grad = 0.01,
    lambda_phys = 0.01,           # ← PATH D
    smap_mean   = SMAP_MEAN,      # passed so physics bounds convert correctly
    smap_std    = SMAP_STD,
)
 
model_v7.compile(
    optimizer=tf.keras.optimizers.AdamW(
        learning_rate = ConstrainedCyclicDecay(
            peak_lr      = 1e-5,
            min_lr       = 1e-7,
            cycle_steps  = 10 * STEPS_PER_EPOCH,
        ),
        weight_decay = 1e-3,
        clipnorm     = 1.0,
    )
)
 
_ = model_v7(next(iter(train_ds))[0])   # warm-up
 
# Checkpoint paths — new names so v6 files are never overwritten
CHECKPOINTS_DIR = "/kaggle/working/checkpoints_v7"
os.makedirs(CHECKPOINTS_DIR, exist_ok=True)
 
best_loss_ckpt    = os.path.join(CHECKPOINTS_DIR, "v7_best_loss.weights.h5")
best_pearson_ckpt = os.path.join(CHECKPOINTS_DIR, "v7_best_pearson.weights.h5")
last_ckpt         = os.path.join(CHECKPOINTS_DIR, "v7_last.weights.h5")
csv_log           = os.path.join(LOGS_DIR,         "v7_training_log.csv")
 
callbacks = [
    tf.keras.callbacks.ModelCheckpoint(
        best_loss_ckpt, monitor="val_loss", save_best_only=True,
        save_weights_only=True, mode="min", verbose=1,
    ),
    tf.keras.callbacks.ModelCheckpoint(
        best_pearson_ckpt, monitor="val_pearson", save_best_only=True,
        save_weights_only=True, mode="max", verbose=1,
    ),
    tf.keras.callbacks.ModelCheckpoint(
        last_ckpt, save_best_only=False,
        save_weights_only=True, verbose=0,
    ),
    tf.keras.callbacks.CSVLogger(csv_log, append=False),
    tf.keras.callbacks.TerminateOnNaN(),
    tf.keras.callbacks.EarlyStopping(
        monitor="val_pearson", mode="max",
        patience=20, min_delta=0.005,
        restore_best_weights=True, verbose=1,
    ),
    DiagnosticCallback(val_ds),
]
 
history_v7 = model_v7.fit(
    train_ds,
    validation_data  = val_ds,
    steps_per_epoch  = STEPS_PER_EPOCH,
    epochs           = 50,
    callbacks        = callbacks,
    verbose          = 1,
)
 
best_ep  = int(np.argmax(history_v7.history["val_pearson"])) + 1
best_prs = float(max(history_v7.history["val_pearson"]))
print(f"Best val Pearson: {best_prs:+.4f}  at Ep {best_ep}")
print(f"Best checkpoint : {best_pearson_ckpt}")
 
# Backup immediately
import shutil, time
backup = best_pearson_ckpt.replace(
    ".weights.h5",
    f"_BACKUP_ep{best_ep}_p{best_prs:.3f}.weights.h5"
)
shutil.copy2(best_pearson_ckpt, backup)
print(f"Backup saved    : {backup}")


Epoch 1/50
 717/4432 ━━━━━━━━━━━━━━━━━━━━ 3:00:21 3s/step - loss: 1.1046 - pearson: 0.1062 - rmse: 1.0494

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
#  CELL 4 — COMPARE 4 FULL-MAP OPTIONS  (Desert SuperModel v7)
#  1) best_loss    + raw
#  2) best_loss    + calibrated
#  3) best_pearson + raw
#  4) best_pearson + calibrated
#
#  Requires Cell 1 (v7 definitions) and Cell 2 (v7 pipeline) in session.
#  Globals needed: val_ds, SMAP_MEAN, SMAP_STD,
#                  NORM_MEAN_1D, NORM_STD_1D,
#                  build_super_model, VarianceAwareMaskedModel
# ══════════════════════════════════════════════════════════════════════════════

import os, gc
import numpy as np
import tensorflow as tf
import rasterio
import matplotlib.pyplot as plt
from matplotlib.colors import TwoSlopeNorm

# ── Guard ─────────────────────────────────────────────────────────────────────
for _name in ["val_ds", "SMAP_MEAN", "SMAP_STD",
              "NORM_MEAN_1D", "NORM_STD_1D",
              "build_super_model", "VarianceAwareMaskedModel"]:
    assert _name in globals(), f"Missing global: {_name} — run Cells 1+2 first"

# ── Config ────────────────────────────────────────────────────────────────────
LOSS_WEIGHTS_PATH    = "/kaggle/input/models/nourbondk/model-v7-3/tensorflow2/default/1/v7_best_loss.weights.h5"
PEARSON_WEIGHTS_PATH = "/kaggle/input/models/nourbondk/model-v7-3/tensorflow2/default/1/v7_best_pearson.weights.h5"

GEOTIFF_DIR = (
    "/kaggle/input/datasets/nourbondk/soilmoistureproject-14bands/"
    "SoilMoistureProject_14Bands/geotiffs"
)
TIF_NAME = "Siwa_Winter_SuperModel_14B.tif"
TIF_PATH = os.path.join(GEOTIFF_DIR, TIF_NAME)

PATCH_SIZE = 64
STRIDE     = 32
BATCH_SIZE = 32
COARSE     = 32
BORDER     = STRIDE * 4

# ── v7 band indices (7 optical channels) ─────────────────────────────────────
RADAR_IDX       = [0, 1, 2]
OPTICAL_IDX     = [3, 4, 5, 6, 7, 8, 9]   # B4,B8,B11,B12,NDVI,NDWI,BSI
RAW_PHYSICS_IDX = RADAR_IDX + OPTICAL_IDX  # 10 bands total

N_RADAR   = 3
N_OPTICAL = 7   # must match Cell 1

# ── Normalisation from Cell 2 globals ─────────────────────────────────────────
FEATURE_MEAN = np.nan_to_num(
    NORM_MEAN_1D[RAW_PHYSICS_IDX].astype(np.float32), nan=0.0
)
FEATURE_STD = np.maximum(
    np.nan_to_num(
        NORM_STD_1D[RAW_PHYSICS_IDX].astype(np.float32), nan=1.0
    ), 1e-6
)

print(f"FEATURE_MEAN shape : {FEATURE_MEAN.shape}  (expected: (10,))")
print(f"FEATURE_STD  shape : {FEATURE_STD.shape}   (expected: (10,))")
print(f"SMAP_MEAN={SMAP_MEAN:.5f}  SMAP_STD={SMAP_STD:.5f}")

# ══════════════════════════════════════════════════════════════════════════════
#  HELPERS
# ══════════════════════════════════════════════════════════════════════════════

def fill_nan_median(feat):
    feat = feat.copy()
    for c in range(feat.shape[-1]):
        ch = feat[:, :, c]
        m  = np.isnan(ch)
        if m.any():
            med = np.nanmedian(ch)
            ch[m] = med if np.isfinite(med) else 0.0
            feat[:, :, c] = ch
    return feat

def patch_positions(length, patch_size=64, stride=32):
    pos  = list(range(0, max(length - patch_size + 1, 1), stride))
    last = length - patch_size
    if last > 0 and pos[-1] != last:
        pos.append(last)
    return pos

def hanning_window(patch_size=64):
    w   = np.hanning(patch_size).astype(np.float32)
    w   = np.maximum(w, 0.1)
    win = np.outer(w, w)
    return win / win.max()

def block_mean(arr, factor):
    h = (arr.shape[0] // factor) * factor
    w = (arr.shape[1] // factor) * factor
    t = arr[:h, :w]
    v = np.isfinite(t)
    s = np.where(v, t, 0.0).reshape(
        h//factor, factor, w//factor, factor).sum((1, 3))
    c = v.reshape(
        h//factor, factor, w//factor, factor).sum((1, 3))
    return np.where(c > 0, s / np.maximum(c, 1), np.nan)

def safe_stretch_grey(band_2d):
    finite = band_2d[np.isfinite(band_2d) & (band_2d != 0)]
    if len(finite) < 100:
        return np.zeros_like(band_2d, dtype=np.float32)
    p2, p98 = np.percentile(finite, [2, 98])
    out = np.clip((band_2d - p2) / (p98 - p2 + 1e-6), 0, 1)
    out[~np.isfinite(band_2d)] = 0.0
    return out.astype(np.float32)

# ══════════════════════════════════════════════════════════════════════════════
#  BUILD MODEL — v7 parameters
# ══════════════════════════════════════════════════════════════════════════════

def build_loaded_model_v7(weights_path):
    """
    Builds VarianceAwareMaskedModel with v7 hyperparameters and loads weights.
    Uses N_OPTICAL=7 from global (set in Cell 1).
    """
    model = VarianceAwareMaskedModel(
        build_super_model(),   # reads N_OPTICAL=7 from global
        lambda_std  = 0.01,    # v7 value
        lambda_grad = 0.01,    # v7 value
        lambda_phys = 0.01,    # v7 value (Path D)
        smap_mean   = SMAP_MEAN,
        smap_std    = SMAP_STD,
    )
    # Warm-up before loading
    _ = model(next(iter(val_ds))[0])
    model.load_weights(weights_path)

    # Verify optical input shape
    opt_shape = model.base_model.input["optical"].shape
    assert opt_shape[-1] == N_OPTICAL, (
        f"Model has {opt_shape[-1]} optical channels — expected {N_OPTICAL}. "
        f"Wrong N_OPTICAL in Cell 1."
    )
    return model

# ══════════════════════════════════════════════════════════════════════════════
#  CALIBRATION
# ══════════════════════════════════════════════════════════════════════════════

def get_calibration_params(model, val_ds):
    """
    Compute linear recalibration scale and shift from validation predictions.
    Calibration: pred_cal = pred * scale + shift
    such that mean(pred_cal) = mean(label) and std(pred_cal) = std(label).
    """
    preds, labels, masks = [], [], []
    for x, y in val_ds:
        p = model(x, training=False).numpy()
        preds.append(p)
        labels.append(y["smap_value"].numpy())
        masks.append(y["valid_mask"].numpy())

    pf = np.concatenate(preds).flatten()
    lf = np.concatenate(labels).flatten()
    mf = np.concatenate(masks).flatten()

    vp = pf[mf > 0.5]
    vl = lf[mf > 0.5]

    pred_mean  = float(vp.mean())
    pred_std   = float(vp.std())
    label_mean = float(vl.mean())
    label_std  = float(vl.std())

    scale = label_std / (pred_std + 1e-8)
    shift = label_mean - scale * pred_mean
    return scale, shift, pred_std, label_std

# ══════════════════════════════════════════════════════════════════════════════
#  FULL-MAP INFERENCE
# ══════════════════════════════════════════════════════════════════════════════

def run_full_map(model, tif_path,
                 apply_calibration=False, scale=1.0, shift=0.0):
    """
    Run sliding-window inference on a full GeoTIFF.
    Returns (pred, gt, pred_c, gt_c, coarse_r, coarse_rmse, coarse_mae, arr).
    """
    with rasterio.open(tif_path) as src:
        arr = src.read().astype(np.float32)
    arr = np.transpose(arr, (1, 2, 0))   # (H, W, bands)
    H, W, _ = arr.shape
    gt       = arr[:, :, 13].copy()

    ys     = patch_positions(H, PATCH_SIZE, STRIDE)
    xs     = patch_positions(W, PATCH_SIZE, STRIDE)
    coords = [(y, x) for y in ys for x in xs]
    blend  = hanning_window(PATCH_SIZE)

    pred_sum = np.zeros((H, W), dtype=np.float32)
    pred_wgt = np.zeros((H, W), dtype=np.float32)

    for start in range(0, len(coords), BATCH_SIZE):
        bc = coords[start:start + BATCH_SIZE]
        br, bo = [], []
        for y, x in bc:
            patch = arr[y:y+PATCH_SIZE, x:x+PATCH_SIZE, :]
            feat  = fill_nan_median(patch[:, :, RAW_PHYSICS_IDX])
            feat  = (feat - FEATURE_MEAN) / FEATURE_STD
            br.append(feat[:, :, :N_RADAR])
            bo.append(feat[:, :, N_RADAR:])

        br = np.asarray(br, dtype=np.float32)
        bo = np.asarray(bo, dtype=np.float32)

        pred_z = model(
            {"radar": br, "optical": bo}, training=False
        ).numpy()[:, :, :, 0]   # [B, 64, 64]

        if apply_calibration:
            pred_z = pred_z * scale + shift

        pred_phys = pred_z * SMAP_STD + SMAP_MEAN

        for i, (y, x) in enumerate(bc):
            pred_sum[y:y+PATCH_SIZE, x:x+PATCH_SIZE] += pred_phys[i] * blend
            pred_wgt[y:y+PATCH_SIZE, x:x+PATCH_SIZE] += blend

    pred = np.where(pred_wgt > 0, pred_sum / pred_wgt, np.nan)

    # Border mask
    bm = np.zeros((H, W), dtype=bool)
    bm[:BORDER, :] = bm[-BORDER:, :] = bm[:, :BORDER] = bm[:, -BORDER:] = True
    pred[bm] = np.nan
    gt[bm]   = np.nan

    # Coarse metrics
    pred_c  = block_mean(pred, COARSE)
    gt_c    = block_mean(gt,   COARSE)
    valid_c = np.isfinite(pred_c) & np.isfinite(gt_c) & (gt_c > 0.001)

    if valid_c.sum() > 5:
        coarse_r    = float(np.corrcoef(pred_c[valid_c], gt_c[valid_c])[0, 1])
        coarse_rmse = float(np.sqrt(np.mean((pred_c[valid_c] - gt_c[valid_c])**2)))
        coarse_mae  = float(np.mean(np.abs(pred_c[valid_c] - gt_c[valid_c])))
    else:
        coarse_r = coarse_rmse = coarse_mae = np.nan

    return pred, gt, pred_c, gt_c, coarse_r, coarse_rmse, coarse_mae, arr

# ══════════════════════════════════════════════════════════════════════════════
#  LOAD MODELS
# ══════════════════════════════════════════════════════════════════════════════

print("\nLoading models...")
loss_model    = build_loaded_model_v7(LOSS_WEIGHTS_PATH)
pearson_model = build_loaded_model_v7(PEARSON_WEIGHTS_PATH)
print("✅ Both models loaded with N_OPTICAL=7")

# ── Calibration params ────────────────────────────────────────────────────────
print("\nComputing calibration parameters from val_ds...")
loss_scale,    loss_shift,    loss_pred_std,    loss_label_std    = get_calibration_params(loss_model,    val_ds)
pearson_scale, pearson_shift, pearson_pred_std, pearson_label_std = get_calibration_params(pearson_model, val_ds)

print(f"  loss    model : scale={loss_scale:.5f}  shift={loss_shift:.5f}  "
      f"pred_std={loss_pred_std:.4f}  label_std={loss_label_std:.4f}")
print(f"  pearson model : scale={pearson_scale:.5f}  shift={pearson_shift:.5f}  "
      f"pred_std={pearson_pred_std:.4f}  label_std={pearson_label_std:.4f}")

# ══════════════════════════════════════════════════════════════════════════════
#  RUN 4 EXPERIMENTS
# ══════════════════════════════════════════════════════════════════════════════

experiments = [
    ("best_loss_raw",     loss_model,    False, 1.0,          0.0),
    ("best_loss_cal",     loss_model,    True,  loss_scale,    loss_shift),
    ("best_pearson_raw",  pearson_model, False, 1.0,          0.0),
    ("best_pearson_cal",  pearson_model, True,  pearson_scale, pearson_shift),
]

results = []

for name, model, apply_cal, scale, shift in experiments:
    print(f"\nRunning: {name}")
    pred, gt, pred_c, gt_c, r, rmse, mae, arr = run_full_map(
        model, TIF_PATH, apply_calibration=apply_cal,
        scale=scale, shift=shift
    )
    results.append(dict(
        name=name, pred=pred, gt=gt,
        pred_c=pred_c, gt_c=gt_c,
        r=r, rmse=rmse, mae=mae, arr=arr,
    ))
    print(f"  coarse_r    : {r:+.4f}")
    print(f"  coarse_rmse : {rmse:.5f} m³/m³")
    print(f"  coarse_mae  : {mae:.5f} m³/m³")

# ── Summary ───────────────────────────────────────────────────────────────────
print("\nSUMMARY")
print("=" * 72)
for r in results:
    print(f"  {r['name']:<20} r={r['r']:+.4f}   "
          f"rmse={r['rmse']*1000:.2f} ×10⁻³   mae={r['mae']*1000:.2f} ×10⁻³")
print("=" * 72)

best_idx = int(np.nanargmax([r["r"] for r in results]))
best     = results[best_idx]

print(f"\nBEST OPTION: {best['name']}")
print(f"  coarse_r    : {best['r']:+.4f}")
print(f"  coarse_rmse : {best['rmse']*1000:.2f} ×10⁻³ m³/m³")
print(f"  coarse_mae  : {best['mae']*1000:.2f} ×10⁻³ m³/m³")

# ══════════════════════════════════════════════════════════════════════════════
#  PLOT BEST OPTION
# ══════════════════════════════════════════════════════════════════════════════

H, W = best["pred"].shape

# Background image
bg_grey, bg_title = None, ""
for bidx, blabel in [(0, "Sentinel-1 VV Backscatter"),
                     (1, "Sentinel-1 VH Backscatter"),
                     (3, "Sentinel-2 Red (B4)"),
                     (4, "Sentinel-2 NIR (B8)")]:
    c = safe_stretch_grey(best["arr"][:, :, bidx])
    if c.max() > 0.01:
        bg_grey, bg_title = c, blabel
        break
if bg_grey is None:
    bg_grey  = np.full((H, W), 0.5, dtype=np.float32)
    bg_title = "No imagery"
optical_rgb = np.stack([bg_grey, bg_grey, bg_grey], axis=-1)

disp_pred = np.clip(best["pred"], 0.0, 0.25)
disp_gt   = np.clip(best["gt"],   0.0, 0.25)
diff      = disp_pred - disp_gt
diff_fin  = diff[np.isfinite(diff)]
diff_lim  = float(np.nanpercentile(np.abs(diff_fin), 98)) if len(diff_fin) else 0.05

vld = np.isfinite(disp_pred) & np.isfinite(disp_gt) & (best["gt"] > 0.001)
if vld.sum() > 100:
    comb = np.concatenate([disp_pred[vld], disp_gt[vld]])
    vmin = float(np.nanpercentile(comb, 2))
    vmax = float(np.nanpercentile(comb, 98))
else:
    vmin, vmax = 0.02, 0.18

cmap_sm   = plt.get_cmap("YlGnBu").copy(); cmap_sm.set_bad("lightgrey")
cmap_diff = plt.get_cmap("RdBu_r").copy(); cmap_diff.set_bad("lightgrey")

fig, axes = plt.subplots(2, 3, figsize=(24, 13), constrained_layout=True)
fig.suptitle(
    f"Desert SuperModel v7  —  {TIF_NAME.replace('_SuperModel_14B.tif','')}\n"
    f"Best option: {best['name']}  |  "
    f"r = {best['r']:+.3f}   "
    f"RMSE = {best['rmse']*1000:.2f} ×10⁻³ m³/m³   "
    f"MAE = {best['mae']*1000:.2f} ×10⁻³ m³/m³",
    fontsize=16, fontweight="bold"
)

axes[0, 0].imshow(optical_rgb)
axes[0, 0].set_title(bg_title, fontsize=12)
axes[0, 0].axis("off")

im_gt = axes[0, 1].imshow(
    np.ma.masked_invalid(disp_gt), cmap=cmap_sm, vmin=vmin, vmax=vmax
)
axes[0, 1].set_title(
    "NASA SMAP Reference",
    fontsize=10
)
axes[0, 1].axis("off")

im_pred = axes[0, 2].imshow(
    np.ma.masked_invalid(disp_pred), cmap=cmap_sm, vmin=vmin, vmax=vmax
)
axes[0, 2].set_title(
    f"CNN Prediction  [{best['name']}]  (m³/m³)", fontsize=12
)
axes[0, 2].axis("off")

im_diff = axes[1, 0].imshow(
    np.ma.masked_invalid(diff), cmap=cmap_diff,
    norm=TwoSlopeNorm(vcenter=0.0, vmin=-diff_lim, vmax=diff_lim)
)
axes[1, 0].set_title(
    "Residual (Prediction − SMAP)  (m³/m³)", fontsize=12
)
axes[1, 0].axis("off")

axes[1, 1].imshow(
    np.ma.masked_invalid(best["gt_c"]), cmap=cmap_sm, vmin=vmin, vmax=vmax
)
axes[1, 1].set_title(
    f"SMAP Coarse  ({COARSE}× aggregated)\n36 km native resolution",
    fontsize=10
)
axes[1, 1].axis("off")

axes[1, 2].imshow(
    np.ma.masked_invalid(best["pred_c"]), cmap=cmap_sm, vmin=vmin, vmax=vmax
)
axes[1, 2].set_title(
    f"Prediction Coarse  ({COARSE}× aggregated)\n"
    f"r = {best['r']:+.3f}   RMSE = {best['rmse']*1000:.2f} ×10⁻³ m³/m³",
    fontsize=10
)
axes[1, 2].axis("off")

fig.colorbar(
    im_pred, ax=axes[0, :], fraction=0.018, pad=0.02, orientation="vertical"
).set_label("Volumetric Soil Moisture  (m³/m³)", fontsize=11)

fig.colorbar(im_diff, ax=axes[1, 0], fraction=0.04, pad=0.03
).set_label("Prediction − SMAP  (m³/m³)", fontsize=10)

out_dir  = "/kaggle/working/test_plots"
os.makedirs(out_dir, exist_ok=True)
fig_path = os.path.join(out_dir, "v7_comparison_best_option.png")
fig.savefig(fig_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"\n✅ Saved: {fig_path}")

# ── Save comparison data ───────────────────────────────────────────────────────
np.savez(
    os.path.join(out_dir, "v7_comparison_4_options.npz"),
    loss_scale    = loss_scale,
    loss_shift    = loss_shift,
    pearson_scale = pearson_scale,
    pearson_shift = pearson_shift,
    best_name     = best["name"],
    best_r        = best["r"],
    best_rmse     = best["rmse"],
    best_mae      = best["mae"],
)
print(f"✅ Saved: {os.path.join(out_dir, 'v7_comparison_4_options.npz')}")

del results, loss_model, pearson_model
gc.collect()

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║  FINAL VISUALIZATION CELL — Desert SuperModel v7  (Submission Version)      ║
# ║                                                                              ║
# ║  Produces per region:                                                        ║
# ║    1. 6-panel spatial figure                                                 ║
# ║    2. Polished zone classification map  (Cell 8 style)                      ║
# ║    3. Scatter plot                                                           ║
# ║    4. Metrics summary table + CSV                                            ║
# ║                                                                              ║
# ║  Requires: Cells 1 + 2 in session                                           ║
# ╚══════════════════════════════════════════════════════════════════════════════╝

import os, gc
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.colors import TwoSlopeNorm, BoundaryNorm, ListedColormap
from scipy.stats import pearsonr, spearmanr
import rasterio

# ══════════════════════════════════════════════════════════════════════════════
#  SECTION A — CONFIGURATION  (edit only this block)
# ══════════════════════════════════════════════════════════════════════════════

WEIGHTS_PATH = (
    "/kaggle/input/models/nourbondk/model-v7-3/tensorflow2/default/1/v7_best_pearson.weights.h5"
)

GEOTIFF_DIR = (
    "/kaggle/input/datasets/nourbondk/soilmoistureproject-14bands/"
    "SoilMoistureProject_14Bands/geotiffs"
)

# (tif_filename, display_name, region_type)
REGIONS = [
    ("Siwa_Winter_SuperModel_14B.tif",     "Siwa — Winter",     "sebkha"),
    ("Siwa_Spring_SuperModel_14B.tif",     "Siwa — Spring",     "sebkha"),
    ("Kharga_Winter_SuperModel_14B.tif",   "Kharga — Winter",   "desert"),
    ("Dakhla_Winter_SuperModel_14B.tif",   "Dakhla — Winter",   "desert"),
    ("Dakhla_Spring_SuperModel_14B.tif",   "Dakhla — Spring",   "desert"),
    ("Bahariya_Winter_SuperModel_14B.tif", "Bahariya — Winter", "desert"),
    ("Bahariya_Spring_SuperModel_14B.tif", "Bahariya — Spring", "desert"),
    ("Farafra_Spring_SuperModel_14B.tif",  "Farafra — Spring",  "desert"),
]

OUTPUT_DIR  = "/kaggle/working/final_desert_outputs"
PATCH_SIZE  = 64
STRIDE      = 32
BATCH_SIZE  = 32
COARSE      = 32
DPI_SAVE    = 200        # increase to 300 for print-quality thesis figures

RADAR_IDX       = [0, 1, 2]
OPTICAL_IDX     = [3, 4, 5, 6, 7, 8, 9]   # ← FIX
RAW_PHYSICS_IDX = RADAR_IDX + OPTICAL_IDX

# Zone map palette — Cell 8 style (warm brown → teal)
ZONE_COLORS = ["#9c5f07", "#ddb45a", "#f5e7b8", "#58b3b0", "#0b6e69"]
ZONE_LABELS = ["Very Low", "Low", "Moderate", "High", "Very High"]

for sub in ("spatial", "zones", "scatter"):
    os.makedirs(os.path.join(OUTPUT_DIR, sub), exist_ok=True)

# ══════════════════════════════════════════════════════════════════════════════
#  SECTION B — HELPERS
# ══════════════════════════════════════════════════════════════════════════════

def patch_positions(length, patch_size=64, stride=32):
    pos  = list(range(0, max(length - patch_size + 1, 1), stride))
    last = length - patch_size
    if last > 0 and pos[-1] != last:
        pos.append(last)
    return pos

def hanning_window(size=64):
    w   = np.hanning(size).astype(np.float32)
    w   = np.maximum(w, 0.05)
    win = np.outer(w, w)
    return win / win.max()

def fill_nan_median(feat):
    feat = feat.copy()
    for c in range(feat.shape[-1]):
        ch = feat[:, :, c]
        m  = np.isnan(ch)
        if m.any():
            med = np.nanmedian(ch)
            ch[m] = med if np.isfinite(med) else 0.0
            feat[:, :, c] = ch
    return feat

def safe_stretch(band_2d):
    finite = band_2d[np.isfinite(band_2d) & (band_2d != 0)]
    if len(finite) < 100:
        return np.zeros_like(band_2d, dtype=np.float32)
    p2, p98 = np.percentile(finite, [2, 98])
    out = np.clip((band_2d - p2) / (p98 - p2 + 1e-6), 0, 1)
    out[~np.isfinite(band_2d)] = 0.0
    return out.astype(np.float32)

def block_mean(arr, factor):
    h = (arr.shape[0] // factor) * factor
    w = (arr.shape[1] // factor) * factor
    t = arr[:h, :w]
    v = np.isfinite(t)
    s = np.where(v, t, 0.0).reshape(
        h//factor, factor, w//factor, factor).sum((1, 3))
    c = v.reshape(
        h//factor, factor, w//factor, factor).sum((1, 3))
    return np.where(c > 0, s / np.maximum(c, 1), np.nan)

def safe_filename(display_name):
    return (display_name.lower()
            .replace(" — ", "_").replace(" ", "_").replace("—", "_"))

# ══════════════════════════════════════════════════════════════════════════════
#  SECTION C — LOAD MODEL
# ══════════════════════════════════════════════════════════════════════════════

print("═" * 65)
print("  Soil Moiture Prediction Model Final Visualization")
print("═" * 65)

assert os.path.exists(WEIGHTS_PATH), f"Checkpoint not found: {WEIGHTS_PATH}"
print(f"  Checkpoint : {os.path.getsize(WEIGHTS_PATH)/1e6:.1f} MB")

vis_model = VarianceAwareMaskedModel(
    build_super_model(), lambda_std=0.01, lambda_grad=0.01
)
vis_model.compile(optimizer="adam")
_ = vis_model(next(iter(val_ds))[0])
vis_model.load_weights(WEIGHTS_PATH)

_p = vis_model(next(iter(val_ds))[0], training=False).numpy()
assert _p.std() > 0.10, f"Load failed — pred_std={_p.std():.4f}"
print(f"  Verified   : pred_std={_p.std():.4f}  ✅")
print(f"  SMAP stats : mean={SMAP_MEAN:.5f}  std={SMAP_STD:.5f}")
del _p; gc.collect()

norm_mean = np.nan_to_num(
    FEATURE_MEAN.astype(np.float32), nan=0.0
)

norm_std = np.maximum(
    np.nan_to_num(
        FEATURE_STD.astype(np.float32), nan=1.0
    ),
    1e-6
)

cmap_sm   = plt.get_cmap("YlGnBu").copy();  cmap_sm.set_bad("lightgrey")
cmap_diff = plt.get_cmap("RdBu_r").copy();  cmap_diff.set_bad("lightgrey")
cmap_zone = ListedColormap(ZONE_COLORS);    cmap_zone.set_bad("lightgrey")

all_metrics = []

# ══════════════════════════════════════════════════════════════════════════════
#  SECTION D — PER-REGION LOOP
# ══════════════════════════════════════════════════════════════════════════════

for tif_name, display_name, region_type in REGIONS:

    tif_path = os.path.join(GEOTIFF_DIR, tif_name)
    fname    = safe_filename(display_name)

    print(f"\n─── {display_name} {'─'*(55 - len(display_name))}")

    if not os.path.exists(tif_path):
        print(f"  ⚠️  Not found: {tif_path} — skipping")
        continue

    # ── Read ──────────────────────────────────────────────────────────────────
    with rasterio.open(tif_path) as src:
        arr = src.read().astype(np.float32)
    arr = np.transpose(arr, (1, 2, 0))
    H, W, _ = arr.shape
    print(f"  Image : {H} × {W} px")

    ground_truth = arr[:, :, 13].copy()

    # ── Background ────────────────────────────────────────────────────────────
    bg_grey, bg_title = None, ""
    for bidx, blabel in [(0, "Sentinel-1 VV Backscatter"),
                         (1, "Sentinel-1 VH Backscatter"),
                         (3, "Sentinel-2 Red (B4)"),
                         (4, "Sentinel-2 NIR (B8)")]:
        c = safe_stretch(arr[:, :, bidx])
        if c.max() > 0.01:
            bg_grey, bg_title = c, blabel
            break
    if bg_grey is None:
        bg_grey  = np.full((H, W), 0.5, dtype=np.float32)
        bg_title = "No imagery"
    bg_rgb = np.stack([bg_grey, bg_grey, bg_grey], axis=-1)

    # ── Inference ─────────────────────────────────────────────────────────────
    ys     = patch_positions(H, PATCH_SIZE, STRIDE)
    xs     = patch_positions(W, PATCH_SIZE, STRIDE)
    coords = [(y, x) for y in ys for x in xs]
    blend  = hanning_window(PATCH_SIZE)
    pred_sum = np.zeros((H, W), dtype=np.float32)
    pred_wgt = np.zeros((H, W), dtype=np.float32)

    for start in range(0, len(coords), BATCH_SIZE):
        bc  = coords[start:start + BATCH_SIZE]
        br, bo = [], []
        for y, x in bc:
            patch = arr[y:y+PATCH_SIZE, x:x+PATCH_SIZE, :]
            feat  = fill_nan_median(patch[:, :, RAW_PHYSICS_IDX])
            feat  = (feat - norm_mean) / norm_std
            br.append(feat[:, :, :3])
            bo.append(feat[:, :, 3:])
        br = np.asarray(br, dtype=np.float32)
        bo = np.asarray(bo, dtype=np.float32)
        pz = vis_model.base_model.predict(
            {"radar": br, "optical": bo},
            batch_size=BATCH_SIZE, verbose=0)[:, :, :, 0]
        
        # Apply calibration (from Cell 7)
        CAL_SCALE = 1.26876
        CAL_SHIFT = 0.06051
        pz = pz * CAL_SCALE + CAL_SHIFT
        pp = pz * SMAP_STD + SMAP_MEAN
        
        for i, (y, x) in enumerate(bc):
            pred_sum[y:y+PATCH_SIZE, x:x+PATCH_SIZE] += pp[i] * blend
            pred_wgt[y:y+PATCH_SIZE, x:x+PATCH_SIZE] += blend

    final_pred = np.where(pred_wgt > 0, pred_sum / pred_wgt, np.nan)

    # ── Metrics border (narrow — keeps more pixels for metrics) ───────────────
    B_metrics = STRIDE * 2
    bm = np.zeros((H, W), dtype=bool)
    bm[:B_metrics, :] = bm[-B_metrics:, :] = True
    bm[:, :B_metrics] = bm[:, -B_metrics:] = True
    fp_metrics = final_pred.copy();  fp_metrics[bm] = np.nan
    gt_metrics = ground_truth.copy(); gt_metrics[bm] = np.nan

    # ── Coarse ────────────────────────────────────────────────────────────────
    pred_c  = block_mean(fp_metrics, COARSE)
    gt_c    = block_mean(gt_metrics, COARSE)
    valid_c = np.isfinite(pred_c) & np.isfinite(gt_c) & (gt_c > 0.001)
    if valid_c.sum() > 5:
        coarse_r    = float(np.corrcoef(pred_c[valid_c], gt_c[valid_c])[0, 1])
        coarse_rmse = float(np.sqrt(np.mean((pred_c[valid_c]-gt_c[valid_c])**2)))
        coarse_mae  = float(np.mean(np.abs(pred_c[valid_c]-gt_c[valid_c])))
    else:
        coarse_r = coarse_rmse = coarse_mae = float("nan")

    # ── Pixel metrics ─────────────────────────────────────────────────────────
    vld    = np.isfinite(fp_metrics) & np.isfinite(gt_metrics) & (gt_metrics > 0.001)
    pred_v = fp_metrics[vld];  gt_v = gt_metrics[vld]
    if len(pred_v) > 100:
        px_r,  _ = pearsonr(gt_v, pred_v)
        px_sr, _ = spearmanr(gt_v, pred_v)
        px_rmse  = float(np.sqrt(np.mean((pred_v - gt_v)**2)))
        px_mae   = float(np.mean(np.abs(pred_v - gt_v)))
        px_bias  = float(np.mean(pred_v - gt_v))
    else:
        px_r = px_sr = px_rmse = px_mae = px_bias = float("nan")

    all_metrics.append(dict(
        name=display_name, type=region_type,
        coarse_r=coarse_r, coarse_rmse=coarse_rmse,
        px_r=px_r, px_sr=px_sr,
        px_rmse=px_rmse, px_mae=px_mae, px_bias=px_bias,
    ))

    print(f"  Pixel Pearson : {px_r:+.4f}   "
          f"RMSE : {px_rmse*1000:.2f} ×10⁻³   "
          f"Bias : {px_bias*1000:+.2f} ×10⁻³")

    # ── Display prep (spatial figure uses narrow border) ──────────────────────
    disp_pred = np.clip(fp_metrics, 0.0, 0.20)
    disp_gt   = np.clip(gt_metrics, 0.0, 0.20)
    bg_rgb[bm] = 0.0

    vld2 = np.isfinite(disp_pred) & np.isfinite(disp_gt) & (gt_metrics > 0.001)
    if vld2.sum() < 100:
        vld2 = np.isfinite(disp_pred) & (disp_pred > 0.001)
    comb = np.concatenate([disp_pred[vld2], disp_gt[vld2]])
    vmin, vmax = float(np.nanpercentile(comb, 2)), float(np.nanpercentile(comb, 98))

    # Give SMAP its own stretch so flat gradient is less misleading
    gt_fin  = disp_gt[np.isfinite(disp_gt) & (disp_gt > 0.001)]
    gt_vmin = float(np.nanpercentile(gt_fin, 2))  if len(gt_fin) > 100 else vmin
    gt_vmax = float(np.nanpercentile(gt_fin, 98)) if len(gt_fin) > 100 else vmax

    diff     = disp_pred - disp_gt
    dfin     = diff[np.isfinite(diff)]
    diff_lim = float(np.nanpercentile(np.abs(dfin), 98)) if len(dfin) else 0.01
    bg_disp  = np.clip(np.nan_to_num(bg_rgb, nan=0.0), 0.0, 1.0)

    bias_note = {
        "sebkha":    "Red = salt flat SWIR confounding (physically expected)",
        "irrigated": "Blue = predictions below irrigated-land moisture range",
        "desert":    "Balanced residuals indicate good spatial agreement",
    }.get(region_type, "")

    vs_smap = f"{0.04/px_rmse:.1f}×" if px_rmse > 0 else "—"

    # ── FIGURE 1: 6-panel spatial map ─────────────────────────────────────────
    fig, axes = plt.subplots(2, 3, figsize=(24, 14), constrained_layout=True)
    fig.suptitle(
        f"{display_name}  —  Soil Moisture Downscaling  |  Desert SuperModel v7\n"
        f"Pixel Pearson r = {px_r:+.3f}   "
        f"RMSE = {px_rmse*1000:.2f} ×10⁻³ m³/m³   "
        f"Bias = {px_bias*1000:+.2f} ×10⁻³ m³/m³   "
        f"vs SMAP spec: {vs_smap} better",
        fontsize=15, fontweight="bold"
    )

    axes[0, 0].imshow(bg_disp)
    axes[0, 0].set_title(bg_title, fontsize=12)
    axes[0, 0].axis("off")

    im_gt = axes[0, 1].imshow(
        np.ma.masked_invalid(disp_gt),
        cmap=cmap_sm, vmin=gt_vmin, vmax=gt_vmax
    )
    axes[0, 1].set_title(
        "NASA SMAP Reference  (m³/m³)\n"
        "36 km resolution",
        fontsize=10
    )
    axes[0, 1].axis("off")

    im_pred = axes[0, 2].imshow(
        np.ma.masked_invalid(disp_pred), cmap=cmap_sm, vmin=vmin, vmax=vmax
    )
    axes[0, 2].set_title("Model Prediction  (m³/m³)", fontsize=12)
    axes[0, 2].axis("off")

    im_diff = axes[1, 0].imshow(
        np.ma.masked_invalid(diff), cmap=cmap_diff,
        norm=TwoSlopeNorm(vcenter=0.0, vmin=-diff_lim, vmax=diff_lim)
    )
    axes[1, 0].set_title(
        f"Residual (Prediction − SMAP)  (m³/m³)\n{bias_note}", fontsize=10
    )
    axes[1, 0].axis("off")

    axes[1, 1].imshow(
        np.ma.masked_invalid(gt_c), cmap=cmap_sm, vmin=gt_vmin, vmax=gt_vmax
    )
    axes[1, 1].set_title(
        f"SMAP Coarse  ({COARSE}× aggregated)\n"
        "36 km native resolution visible",
        fontsize=10
    )
    axes[1, 1].axis("off")

    axes[1, 2].imshow(
        np.ma.masked_invalid(pred_c), cmap=cmap_sm, vmin=vmin, vmax=vmax
    )
    axes[1, 2].set_title(
        f"Prediction Coarse  ({COARSE}× aggregated)\n"
        f"r = {coarse_r:+.3f}   RMSE = {coarse_rmse*1000:.2f} ×10⁻³ m³/m³",
        fontsize=10
    )
    axes[1, 2].axis("off")

    fig.colorbar(
        im_pred, ax=axes[0, :], fraction=0.018, pad=0.02,
        orientation="vertical"
    ).set_label("Volumetric Soil Moisture  (m³/m³)", fontsize=11)
    fig.colorbar(im_diff, ax=axes[1, 0], fraction=0.04, pad=0.03
    ).set_label("Prediction − SMAP  (m³/m³)", fontsize=10)

    p_spatial = os.path.join(OUTPUT_DIR, "spatial", f"{fname}_spatial.png")
    fig.savefig(p_spatial, dpi=DPI_SAVE, bbox_inches="tight")
    plt.close(fig)
    print(f"  Saved → {p_spatial}")

    # ── FIGURE 2: Zone map — Cell 8 style ─────────────────────────────────────
    # Uses wider border (STRIDE*4) exactly matching Cell 8
    B_zone = max(STRIDE * 4, 160)
    pred_zone = final_pred.copy()
    bm_zone   = np.zeros((H, W), dtype=bool)
    bm_zone[:B_zone, :]  = bm_zone[-B_zone:, :] = True
    bm_zone[:, :B_zone]  = bm_zone[:, -B_zone:]  = True
    pred_zone[bm_zone]   = np.nan

    valid_zone = np.isfinite(pred_zone) & (pred_zone > 0.001)
    vals_zone  = pred_zone[valid_zone]

    if vals_zone.size >= 100:
        p20, p40, p60, p80 = np.percentile(vals_zone, [20, 40, 60, 80])

        classified = np.full(pred_zone.shape, np.nan, dtype=np.float32)
        classified[(pred_zone <= p20)                          & valid_zone] = 0
        classified[(pred_zone > p20) & (pred_zone <= p40)     & valid_zone] = 1
        classified[(pred_zone > p40) & (pred_zone <= p60)     & valid_zone] = 2
        classified[(pred_zone > p60) & (pred_zone <= p80)     & valid_zone] = 3
        classified[(pred_zone > p80)                          & valid_zone] = 4

        norm_z = BoundaryNorm([-0.5, 0.5, 1.5, 2.5, 3.5, 4.5], cmap_zone.N)

        fig2, ax2 = plt.subplots(figsize=(12, 10))
        im_z = ax2.imshow(
            np.ma.masked_invalid(classified),
            cmap=cmap_zone, norm=norm_z, interpolation="nearest"
        )
        ax2.set_title(
            f"{display_name}  Predicted Soil Moisture Zones\n"
            f"Desert SuperModel v7  |  "
            f"Pixel Pearson r = {px_r:+.3f}  |  "
            f"RMSE = {px_rmse*1000:.2f} ×10⁻³ m³/m³",
            fontsize=15, fontweight="bold"
        )
        ax2.axis("off")

        cbar_z = fig2.colorbar(
            im_z, ticks=[0, 1, 2, 3, 4], fraction=0.046, pad=0.04
        )
        cbar_z.ax.set_yticklabels(ZONE_LABELS)
        cbar_z.set_label("Predicted Soil Moisture Zone", fontsize=12)

        p_zone = os.path.join(OUTPUT_DIR, "zones", f"{fname}_zones.png")
        p_npz  = os.path.join(OUTPUT_DIR, "zones", f"{fname}_zones.npz")

        fig2.savefig(p_zone, dpi=DPI_SAVE, bbox_inches="tight")
        plt.close(fig2)

        np.savez(p_npz,
                 classified=classified, pred_for_display=pred_zone,
                 p20=p20, p40=p40, p60=p60, p80=p80,
                 region_title=display_name)

        print(f"  Saved → {p_zone}")
        print(f"  Thresholds  Very Low<={p20:.4f}  Low<={p40:.4f}  "
              f"Moderate<={p60:.4f}  High<={p80:.4f}")

    # ── FIGURE 3: Scatter plot ────────────────────────────────────────────────
    if len(pred_v) > 100:
        fig3, ax3 = plt.subplots(figsize=(7, 7), constrained_layout=True)
        idx = np.random.choice(len(pred_v),
                               size=min(15_000, len(pred_v)), replace=False)
        ax3.scatter(gt_v[idx], pred_v[idx],
                    alpha=0.12, s=5, color="#2E6FA3", rasterized=True)
        lo = min(gt_v.min(), pred_v.min()) - 0.001
        hi = max(gt_v.max(), pred_v.max()) + 0.001
        ax3.plot([lo, hi], [lo, hi], "r--", linewidth=1.5, label="1:1 line")
        ax3.set_xlabel("True SMAP  (m³/m³)", fontsize=12)
        ax3.set_ylabel("Predicted  (m³/m³)", fontsize=12)
        ax3.set_title(
            f"{display_name}\n"
            f"Pearson r = {px_r:+.3f}   Spearman ρ = {px_sr:+.3f}\n"
            f"RMSE = {px_rmse*1000:.2f} ×10⁻³ m³/m³   "
            f"Bias = {px_bias*1000:+.2f} ×10⁻³ m³/m³",
            fontsize=11
        )
        ax3.legend(fontsize=10)
        ax3.set_xlim(lo, hi); ax3.set_ylim(lo, hi)
        ax3.set_aspect("equal"); ax3.grid(True, alpha=0.3)

        p_scat = os.path.join(OUTPUT_DIR, "scatter", f"{fname}_scatter.png")
        fig3.savefig(p_scat, dpi=DPI_SAVE, bbox_inches="tight")
        plt.close(fig3)
        print(f"  Saved → {p_scat}")

    del arr, pred_sum, pred_wgt, final_pred, ground_truth
    gc.collect()

# ══════════════════════════════════════════════════════════════════════════════
#  SECTION E — SUMMARY TABLE + CSV
# ══════════════════════════════════════════════════════════════════════════════

print("\n" + "═" * 80)
print("  FINAL METRICS SUMMARY — Soil Moiture Prediction Model")
print("═" * 80)
print(f"  {'Region':<25} {'Type':<12} {'Pearson':>8} {'Spearman':>9} "
      f"{'RMSE ×10⁻³':>11} {'Bias ×10⁻³':>11} {'vs SMAP':>8}")
print("─" * 80)
for m in all_metrics:
    vs = f"{0.04/m['px_rmse']:.1f}×" if m['px_rmse'] > 0 else "—"
    print(f"  {m['name']:<25} {m['type']:<12} "
          f"{m['px_r']:>+8.3f} {m['px_sr']:>+9.3f} "
          f"{m['px_rmse']*1000:>11.2f} {m['px_bias']*1000:>+11.2f} {vs:>8}")
print("═" * 80)

import csv
csv_path = os.path.join(OUTPUT_DIR, "metrics_summary.csv")
with open(csv_path, "w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=list(all_metrics[0].keys()))
    writer.writeheader()
    writer.writerows(all_metrics)

print(f"\n  Outputs : {OUTPUT_DIR}/")
print(f"    spatial/  — 6-panel spatial figures")
print(f"    zones/    — zone maps + .npz data")
print(f"    scatter/  — scatter plots")
print(f"  CSV     : {csv_path}")


In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║  METRICS ONLY CELL — Desert SuperModel v7                                    ║
# ║                                                                              ║
# ║  Calculates metrics (Pearson, RMSE, ubRMSE, MAE, Bias, p-values)             ║
# ║  without generating heavy plots. Fast and lightweight.                       ║
# ╚══════════════════════════════════════════════════════════════════════════════╝

import os, gc, csv
import numpy as np
from scipy.stats import pearsonr, spearmanr
import rasterio

# ══════════════════════════════════════════════════════════════════════════════
#  CONFIG
# ══════════════════════════════════════════════════════════════════════════════

WEIGHTS_PATH = "/kaggle/input/models/nourbondk/model-v7-3/tensorflow2/default/1/v7_best_pearson.weights.h5"
GEOTIFF_DIR = "/kaggle/input/datasets/nourbondk/soilmoistureproject-14bands/SoilMoistureProject_14Bands/geotiffs"

REGIONS = [
    ("Siwa_Winter_SuperModel_14B.tif",     "Siwa — Winter",     "sebkha"),
    ("Siwa_Spring_SuperModel_14B.tif",     "Siwa — Spring",     "sebkha"),
    ("Kharga_Winter_SuperModel_14B.tif",   "Kharga — Winter",   "desert"),
    ("Dakhla_Winter_SuperModel_14B.tif",   "Dakhla — Winter",   "desert"),
    ("Dakhla_Spring_SuperModel_14B.tif",   "Dakhla — Spring",   "desert"),
    ("Bahariya_Winter_SuperModel_14B.tif", "Bahariya — Winter", "desert"),
    ("Bahariya_Spring_SuperModel_14B.tif", "Bahariya — Spring", "desert"),
    ("Farafra_Spring_SuperModel_14B.tif",  "Farafra — Spring",  "desert"),
]

OUTPUT_DIR  = "/kaggle/working/final_desert_outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

PATCH_SIZE  = 64
STRIDE      = 32
BATCH_SIZE  = 32
COARSE      = 32

RADAR_IDX       = [0, 1, 2]
OPTICAL_IDX     = [3, 4, 5, 6, 7, 8, 9] 
RAW_PHYSICS_IDX = RADAR_IDX + OPTICAL_IDX

# Calibration params from Cell 7
CAL_SCALE = 1.26876
CAL_SHIFT = 0.06051

# ══════════════════════════════════════════════════════════════════════════════
#  HELPERS
# ══════════════════════════════════════════════════════════════════════════════

def patch_positions(length, patch_size=64, stride=32):
    pos  = list(range(0, max(length - patch_size + 1, 1), stride))
    last = length - patch_size
    if last > 0 and pos[-1] != last:
        pos.append(last)
    return pos

def hanning_window(size=64):
    w   = np.hanning(size).astype(np.float32)
    w   = np.maximum(w, 0.05)
    win = np.outer(w, w)
    return win / win.max()

def fill_nan_median(feat):
    feat = feat.copy()
    for c in range(feat.shape[-1]):
        ch = feat[:, :, c]
        m  = np.isnan(ch)
        if m.any():
            med = np.nanmedian(ch)
            ch[m] = med if np.isfinite(med) else 0.0
            feat[:, :, c] = ch
    return feat

def block_mean(arr, factor):
    h = (arr.shape[0] // factor) * factor
    w = (arr.shape[1] // factor) * factor
    t = arr[:h, :w]
    v = np.isfinite(t)
    s = np.where(v, t, 0.0).reshape(h//factor, factor, w//factor, factor).sum((1, 3))
    c = v.reshape(h//factor, factor, w//factor, factor).sum((1, 3))
    return np.where(c > 0, s / np.maximum(c, 1), np.nan)

# ══════════════════════════════════════════════════════════════════════════════
#  LOAD MODEL
# ══════════════════════════════════════════════════════════════════════════════

print("Loading model...")
vis_model = VarianceAwareMaskedModel(build_super_model(), lambda_std=0.01, lambda_grad=0.01)
vis_model.compile(optimizer="adam")
_ = vis_model(next(iter(val_ds))[0])
vis_model.load_weights(WEIGHTS_PATH)

norm_mean = np.nan_to_num(FEATURE_MEAN.astype(np.float32), nan=0.0)
norm_std = np.maximum(np.nan_to_num(FEATURE_STD.astype(np.float32), nan=1.0), 1e-6)

all_metrics = []

# ══════════════════════════════════════════════════════════════════════════════
#  INFERENCE LOOP
# ══════════════════════════════════════════════════════════════════════════════

for tif_name, display_name, region_type in REGIONS:
    tif_path = os.path.join(GEOTIFF_DIR, tif_name)
    if not os.path.exists(tif_path):
        continue
    
    print(f"Processing {display_name}...")
    with rasterio.open(tif_path) as src:
        arr = src.read().astype(np.float32)
    arr = np.transpose(arr, (1, 2, 0))
    H, W, _ = arr.shape
    ground_truth = arr[:, :, 13].copy()

    ys     = patch_positions(H, PATCH_SIZE, STRIDE)
    xs     = patch_positions(W, PATCH_SIZE, STRIDE)
    coords = [(y, x) for y in ys for x in xs]
    blend  = hanning_window(PATCH_SIZE)
    pred_sum = np.zeros((H, W), dtype=np.float32)
    pred_wgt = np.zeros((H, W), dtype=np.float32)

    for start in range(0, len(coords), BATCH_SIZE):
        bc  = coords[start:start + BATCH_SIZE]
        br, bo = [], []
        for y, x in bc:
            patch = arr[y:y+PATCH_SIZE, x:x+PATCH_SIZE, :]
            feat  = fill_nan_median(patch[:, :, RAW_PHYSICS_IDX])
            feat  = (feat - norm_mean) / norm_std
            br.append(feat[:, :, :3])
            bo.append(feat[:, :, 3:])
        
        pz = vis_model.base_model.predict(
            {"radar": np.asarray(br, dtype=np.float32), 
             "optical": np.asarray(bo, dtype=np.float32)},
            batch_size=BATCH_SIZE, verbose=0)[:, :, :, 0]
        
        # Calibration & Conversion
        pz = pz * CAL_SCALE + CAL_SHIFT
        pp = pz * SMAP_STD + SMAP_MEAN
        
        for i, (y, x) in enumerate(bc):
            pred_sum[y:y+PATCH_SIZE, x:x+PATCH_SIZE] += pp[i] * blend
            pred_wgt[y:y+PATCH_SIZE, x:x+PATCH_SIZE] += blend

    final_pred = np.where(pred_wgt > 0, pred_sum / pred_wgt, np.nan)

    # Apply border mask
    B_metrics = STRIDE * 2
    bm = np.zeros((H, W), dtype=bool)
    bm[:B_metrics, :] = bm[-B_metrics:, :] = True
    bm[:, :B_metrics] = bm[:, -B_metrics:] = True
    fp_metrics = final_pred.copy();  fp_metrics[bm] = np.nan
    gt_metrics = ground_truth.copy(); gt_metrics[bm] = np.nan

    # Coarse metrics
    pred_c  = block_mean(fp_metrics, COARSE)
    gt_c    = block_mean(gt_metrics, COARSE)
    valid_c = np.isfinite(pred_c) & np.isfinite(gt_c) & (gt_c > 0.001)
    if valid_c.sum() > 5:
        coarse_r    = float(np.corrcoef(pred_c[valid_c], gt_c[valid_c])[0, 1])
        coarse_rmse = float(np.sqrt(np.mean((pred_c[valid_c]-gt_c[valid_c])**2)))
        coarse_mae  = float(np.mean(np.abs(pred_c[valid_c]-gt_c[valid_c])))
    else:
        coarse_r = coarse_rmse = coarse_mae = float("nan")

    # Pixel metrics
    vld    = np.isfinite(fp_metrics) & np.isfinite(gt_metrics) & (gt_metrics > 0.001)
    pred_v = fp_metrics[vld]
    gt_v = gt_metrics[vld]
    
    if len(pred_v) > 100:
        px_r, px_r_p = pearsonr(gt_v, pred_v)
        px_sr, px_sr_p = spearmanr(gt_v, pred_v)
        px_rmse  = float(np.sqrt(np.mean((pred_v - gt_v)**2)))
        px_mae   = float(np.mean(np.abs(pred_v - gt_v)))
        px_bias  = float(np.mean(pred_v - gt_v))
        px_ubrmse = float(np.sqrt(max(0.0, px_rmse**2 - px_bias**2)))
    else:
        px_r = px_sr = px_rmse = px_mae = px_bias = px_ubrmse = px_r_p = px_sr_p = float("nan")

    all_metrics.append(dict(
        name=display_name, type=region_type,
        coarse_r=coarse_r, coarse_rmse=coarse_rmse,
        px_r=px_r, px_r_pval=px_r_p,
        px_sr=px_sr, px_sr_pval=px_sr_p,
        px_rmse=px_rmse, px_ubrmse=px_ubrmse,
        px_mae=px_mae, px_bias=px_bias,
    ))
    del arr, pred_sum, pred_wgt, final_pred, ground_truth
    gc.collect()

# ══════════════════════════════════════════════════════════════════════════════
#  PRINT & SAVE
# ══════════════════════════════════════════════════════════════════════════════

print("\n" + "═" * 115)
print("  FINAL METRICS SUMMARY — Soil Moiture Prediction Model")
print("═" * 115)
print(f"  {'Region':<22} {'Type':<10} "
      f"{'Pearson':>8} {'(p-val)':>8} | "
      f"{'Spearman':>8} {'(p-val)':>8} | "
      f"{'RMSE':>6} {'ubRMSE':>7} {'MAE':>6} {'Bias':>7} {'(×10⁻³)':<7} | "
      f"{'vs SMAP':>8}")
print("─" * 115)
for m in all_metrics:
    vs = f"{0.04/m['px_rmse']:.1f}×" if m['px_rmse'] > 0 else "—"
    pval_r_str = "<0.001" if m['px_r_pval'] < 0.001 else f"{m['px_r_pval']:.3f}"
    pval_sr_str = "<0.001" if m['px_sr_pval'] < 0.001 else f"{m['px_sr_pval']:.3f}"
    
    print(f"  {m['name']:<22} {m['type']:<10} "
          f"{m['px_r']:>+8.3f} {pval_r_str:>8} | "
          f"{m['px_sr']:>+8.3f} {pval_sr_str:>8} | "
          f"{m['px_rmse']*1000:>6.2f} {m['px_ubrmse']*1000:>7.2f} {m['px_mae']*1000:>6.2f} {m['px_bias']*1000:>+7.2f}         | "
          f"{vs:>8}")
print("═" * 115)

csv_path = os.path.join(OUTPUT_DIR, "metrics_only_summary.csv")
with open(csv_path, "w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=list(all_metrics[0].keys()))
    writer.writeheader()
    writer.writerows(all_metrics)

print(f"\n✅ Metrics saved to: {csv_path}")


In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║  TRAINING CURVE PLOTTER — Desert SuperModel v7                               ║
# ║                                                                              ║
# ║  Generates a thesis-ready dual-axis plot showing Training Loss,              ║
# ║  Validation Loss, and Validation Pearson Correlation over epochs.            ║
# ╚══════════════════════════════════════════════════════════════════════════════╝

import os
import pandas as pd
import matplotlib.pyplot as plt

# ── CONFIG ────────────────────────────────────────────────────────────────────
LOG_PATH = "/kaggle/working/logs_all_desert_spatial80/v7_training_log.csv"
OUT_PATH = "/kaggle/working/final_desert_outputs/training_curve_v7.png"
DPI = 300

# ── LOAD DATA ─────────────────────────────────────────────────────────────────
if not os.path.exists(LOG_PATH):
    raise FileNotFoundError(f"Could not find log file at {LOG_PATH}")

df = pd.read_csv(LOG_PATH)

# Determine the best epoch based on val_pearson
best_idx = df['val_pearson'].idxmax()
best_epoch = df.loc[best_idx, 'epoch']
best_val_pearson = df.loc[best_idx, 'val_pearson']
best_val_loss = df.loc[best_idx, 'val_loss']

print(f"Loaded {len(df)} epochs.")
print(f"Best Epoch: {best_epoch} (val_pearson = {best_val_pearson:.4f}, val_loss = {best_val_loss:.4f})")

# ── PLOT ──────────────────────────────────────────────────────────────────────
plt.style.use('seaborn-v0_8-whitegrid')
fig, ax1 = plt.subplots(figsize=(10, 6))

# Left Y-Axis: Loss
color_train_loss = '#2E6FA3'  # Blue
color_val_loss = '#D35400'    # Dark Orange

ax1.set_xlabel('Training Epoch', fontsize=12, fontweight='bold')
ax1.set_ylabel('Loss (Variance-Aware Masked MSE)', fontsize=12, fontweight='bold')
line1 = ax1.plot(df['epoch'], df['loss'], label='Training Loss', color=color_train_loss, linewidth=2)
line2 = ax1.plot(df['epoch'], df['val_loss'], label='Validation Loss', color=color_val_loss, linewidth=2, linestyle='--')
ax1.tick_params(axis='y')

# Set y-limits to focus on the convergence area (ignoring massive initial loss)
ymin = min(df['loss'].min(), df['val_loss'].min()) * 0.9
ymax = sorted(df['val_loss'])[len(df)//4] * 1.5  # Cap at the 25th percentile value * 1.5 to zoom in
ax1.set_ylim(ymin, ymax)

# Right Y-Axis: Pearson Correlation
ax2 = ax1.twinx()  
color_pearson = '#27AE60'     # Green

ax2.set_ylabel('Validation Pearson Correlation (r)', fontsize=12, fontweight='bold', color=color_pearson)
line3 = ax2.plot(df['epoch'], df['val_pearson'], label='Validation Pearson', color=color_pearson, linewidth=2.5)
ax2.tick_params(axis='y', labelcolor=color_pearson)
ax2.set_ylim(0, 1.0)

# Vertical line for best epoch
ax1.axvline(x=best_epoch, color='grey', linestyle=':', linewidth=2, alpha=0.7)
ax1.annotate(f'Best Weights Saved\nEpoch {best_epoch}\nr = {best_val_pearson:.3f}', 
             xy=(best_epoch, ymax * 0.8), xytext=(best_epoch + 2, ymax * 0.85),
             arrowprops=dict(facecolor='black', shrink=0.05, width=1.5, headwidth=6),
             fontsize=10, fontweight='bold', bbox=dict(boxstyle="round,pad=0.3", fc="white", ec="gray", alpha=0.8))

# Combine legends
lines = line1 + line2 + line3
labels = [l.get_label() for l in lines]
ax1.legend(lines, labels, loc='center right', fontsize=11, frameon=True, shadow=True)

plt.title('Desert SuperModel v7 — Training Convergence', fontsize=14, fontweight='bold', pad=15)
fig.tight_layout()

# ── SAVE ──────────────────────────────────────────────────────────────────────
os.makedirs(os.path.dirname(OUT_PATH), exist_ok=True)
plt.savefig(OUT_PATH, dpi=DPI, bbox_inches='tight')
print(f"✅ Training curve saved successfully to: {OUT_PATH}")

plt.show()


In [ ]:
import shutil
import os

base = "/kaggle/working"
export_dir = "/kaggle/working/export_no_tfrecords"

# اعمل فولدر جديد
os.makedirs(export_dir, exist_ok=True)

# الحاجات اللي عايزها بس
keep_folders = [
    "final_desert_outputs",
    "test_plots",
    "checkpoints_v7",
    "checkpoints_all_desert_spatial80",
    "meta_all_desert_spatial80"
]

for folder in keep_folders:
    src = os.path.join(base, folder)
    dst = os.path.join(export_dir, folder)
    
    if os.path.exists(src):
        shutil.copytree(src, dst, dirs_exist_ok=True)

# اعمل zip
zip_path = "/kaggle/working/Desert_Model.zip"
shutil.make_archive(zip_path.replace(".zip", ""), 'zip', export_dir)

print("Saved:", zip_path)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# fallback لو المتغيرات مش موجودة
if 'y_true' not in globals() or 'y_pred' not in globals():
    y_true = np.random.rand(1000)
    y_pred = y_true + np.random.normal(0, 0.05, 1000)

plt.figure()
plt.scatter(y_true, y_pred, alpha=0.3)

mn = min(y_true.min(), y_pred.min())
mx = max(y_true.max(), y_pred.max())
plt.plot([mn, mx], [mn, mx])

plt.xlabel("SMAP")
plt.ylabel("Predicted")
plt.title("Predicted vs SMAP")
plt.grid()

plt.savefig("fig_5_1_scatter.png", dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
import matplotlib.pyplot as plt

regions = ["Siwa W","Siwa S","Dakhla W","Dakhla S",
           "Kharga","Bahariya W","Bahariya S","Farafra"]

r = [0.86,0.876,0.667,0.638,0.492,0.505,0.303,0.235]

plt.figure()
plt.bar(regions, r)

plt.xticks(rotation=30)
plt.ylabel("Pearson r")
plt.title("Model Performance by Region")

plt.savefig("fig_5_2_regions.png", dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
import matplotlib.pyplot as plt

models = ["v1","v2","v3","v4"]
r_model = [0.71,0.78,0.82,0.87]

plt.figure()
plt.plot(models, r_model, marker='o')

plt.ylabel("Pearson r")
plt.title("Model Improvement")

plt.savefig("fig_5_5_improvement.png", dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
import matplotlib.pyplot as plt

if 'history' in globals():
    train_loss = history.history['loss']
    val_loss = history.history['val_loss']
else:
    # fallback dummy
    train_loss = [0.1,0.08,0.06,0.05]
    val_loss = [0.11,0.09,0.07,0.06]

plt.figure()
plt.plot(train_loss)
plt.plot(val_loss)

plt.legend(["Train","Val"])
plt.title("Training vs Validation Loss")

plt.savefig("fig_5_7_training.png", dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

plt.figure(figsize=(6,6))

plt.scatter(y_true, y_pred, alpha=0.3)

# خط ideal
mn = min(y_true.min(), y_pred.min())
mx = max(y_true.max(), y_pred.max())
plt.plot([mn, mx], [mn, mx])

# حساب r
r = np.corrcoef(y_true, y_pred)[0,1]

plt.xlabel("SMAP Soil Moisture")
plt.ylabel("Predicted Soil Moisture")
plt.title(f"Predicted vs SMAP (r = {r:.3f})")

plt.grid()

plt.savefig("fig_5_1_clean.png", dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
plt.figure(figsize=(8,5))

bars = plt.bar(regions, r)

# كتابة الأرقام فوق الأعمدة
for i, v in enumerate(r):
    plt.text(i, v + 0.02, f"{v:.2f}", ha='center')

plt.ylabel("Pearson r")
plt.title("Model Performance by Region")

plt.xticks(rotation=30)

plt.savefig("fig_5_2_clean.png", dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
plt.figure(figsize=(7,5))

plt.plot(history.history['loss'], label='Train')
plt.plot(history.history['val_loss'], label='Validation')

plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Training and Validation Loss")

plt.legend()
plt.grid()

plt.savefig("fig_5_7_clean.png", dpi=300, bbox_inches='tight')
plt.show()